# Clasificador de Madurez de Platanos - Notebook de PRUEBA

Este notebook permite **probar los modelos ya entrenados sin necesidad de re-entrenar**.
Carga los modelos directamente desde el repositorio de GitHub y ofrece una interfaz
grafica para subir una foto de un platano y obtener su clasificacion.

**Uso:** ejecutar las celdas en orden (Entorno de ejecucion -> Ejecutar todas).
No requiere descargar el dataset ni entrenar. Para clasificar con YOLO se recomienda
activar la GPU (Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU), aunque
tambien funciona en CPU para probar imagenes sueltas.

Trabajo Final - Procesamiento de Imagenes (1ACC0235) - UPC

## Bloque 1: Instalar librerias y descargar los modelos desde GitHub

In [ ]:
# === BLOQUE 1: Setup - instalar librerias y descargar modelos ===
import subprocess
print("Instalando librerias...")
subprocess.run(["pip", "install", "-q", "ultralytics"])

import os
import urllib.request

# URL base del repositorio (archivos crudos). Ajustar el usuario si es distinto.
URL_BASE = "https://github.com/Lisztzz/1ACC0235-TP-TF-2026-01/raw/main/code/"

# Descargar los dos modelos entrenados
modelos = {
    "modelo_platanos_yolo.pt": URL_BASE + "modelo_platanos_yolo.pt",
    "modelo_platanos_cnn.keras": URL_BASE + "modelo_platanos_cnn.keras",
}

for nombre, url in modelos.items():
    if not os.path.exists(nombre):
        print(f"Descargando {nombre}...")
        try:
            urllib.request.urlretrieve(url, nombre)
            print(f"  OK: {nombre} ({os.path.getsize(nombre)/1024/1024:.1f} MB)")
        except Exception as e:
            print(f"  ERROR descargando {nombre}: {e}")
            print("  Verifica que el modelo este subido al repositorio en la carpeta code/")
    else:
        print(f"{nombre} ya esta descargado.")

print("\nSetup completo.")

## Bloque 2: Cargar los modelos y definir variables

In [ ]:
# === BLOQUE 2: Cargar modelos y variables globales ===
from ultralytics import YOLO
from tensorflow.keras.models import load_model

# Cargar YOLO (modelo principal, el de mejor desempeno)
modeloYOLO = YOLO("modelo_platanos_yolo.pt")
print("Modelo YOLO11 cargado.")

# Cargar CNN (opcional, para comparar)
modeloCNN = load_model("modelo_platanos_cnn.keras")
print("Modelo CNN (MobileNetV2) cargado.")

# Variables globales
clases = ["unripe", "ripe", "overripe", "rotten"]
nombresEspanol = {"unripe": "Verde", "ripe": "Maduro",
                  "overripe": "Sobremaduro", "rotten": "Podrido"}

# Informacion logistica por clase
infoLogistica = {
    "unripe": {"apto": "No apto para consumo inmediato (requiere maduracion)",
               "ventana": "7-14 dias", "mercado": "Mercados lejanos / Exportacion",
               "obs": "Resistente al transporte. Puede madurar en destino."},
    "ripe": {"apto": "Apto para consumo inmediato",
             "ventana": "3-7 dias", "mercado": "Mercados regionales / Consumo directo",
             "obs": "Estado optimo para venta al consumidor."},
    "overripe": {"apto": "Apto, preferible para procesamiento industrial",
                 "ventana": "1-2 dias", "mercado": "Mercado local / Industria (pures, harina)",
                 "obs": "Alto riesgo de perdida. Comercializar con urgencia."},
    "rotten": {"apto": "NO apto para consumo",
               "ventana": "0 dias (desecho)", "mercado": "Ninguno",
               "obs": "Separar del lote para evitar contaminacion."}
}
print("\nVariables definidas. Listo para clasificar.")

## Bloque 3: Interfaz grafica - Clasificar con YOLO11 (modelo recomendado)

In [ ]:
# === BLOQUE 3: Interfaz grafica con YOLO11 ===
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from PIL import Image
import io
import matplotlib.pyplot as plt

titulo = widgets.HTML(
    "<h2 style='color:#333'>Clasificador de Madurez de Platanos (YOLO11)</h2>"
    "<p style='color:#555'>Sube una foto de un platano y presiona Clasificar.</p>"
)
cargador = widgets.FileUpload(accept="image/*", multiple=False)
boton = widgets.Button(description="Clasificar", button_style="success",
                       icon="check", layout=widgets.Layout(width="200px", height="40px"))
salida = widgets.Output()

def procesar(b):
    with salida:
        clear_output()
        if len(cargador.value) == 0:
            print("Primero sube una imagen, luego presiona Clasificar.")
            return
        try:
            valor = cargador.value
            contenido = (list(valor.values())[0]["content"] if isinstance(valor, dict)
                         else valor[0]["content"])
        except Exception as e:
            print(f"Error leyendo el archivo: {e}")
            return

        imgPil = Image.open(io.BytesIO(contenido)).convert("RGB")
        imgArray = np.array(imgPil)

        resultado = modeloYOLO.predict(imgArray, verbose=False)
        probs = resultado[0].probs
        idxClase = probs.top1
        claseIngles = resultado[0].names[idxClase]
        claseEspanol = nombresEspanol[claseIngles]
        confianza = probs.top1conf.item() * 100
        info = infoLogistica[claseIngles]

        plt.figure(figsize=(5, 5))
        plt.imshow(imgPil); plt.axis("off")
        plt.title(f"Modelo YOLO11: {claseEspanol} ({confianza:.1f}%)", fontsize=13)
        plt.show()

        print("=" * 52)
        print(f"  MODELO YOLO11 -> {claseEspanol}  (confianza {confianza:.1f}%)")
        print("=" * 52)
        print(f"\n  Apto para consumo?  {info['apto']}")
        print(f"  Ventana de transporte: {info['ventana']}")
        print(f"  Mercado recomendado:   {info['mercado']}")
        print(f"  Observacion:           {info['obs']}")
        print(f"\n  Probabilidades por clase:")
        todasProbs = probs.data.tolist()
        for i, nombreYolo in resultado[0].names.items():
            barra = "#" * int(todasProbs[i] * 25)
            print(f"    {nombresEspanol[nombreYolo]:14s}: {todasProbs[i]*100:5.1f}%  {barra}")

boton.on_click(procesar)
display(titulo, cargador, boton, salida)

## Bloque 4 (opcional): Interfaz grafica - Clasificar con CNN (MobileNetV2)

Este bloque es opcional, para comparar el resultado del CNN con el de YOLO11.

In [ ]:
# === BLOQUE 4 (opcional): Interfaz grafica con CNN ===
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from PIL import Image
import io
import cv2
import matplotlib.pyplot as plt

tituloCNN = widgets.HTML(
    "<h2 style='color:#333'>Clasificador de Madurez de Platanos (CNN MobileNetV2)</h2>"
    "<p style='color:#555'>Sube una foto de un platano y presiona Clasificar.</p>"
)
cargadorCNN = widgets.FileUpload(accept="image/*", multiple=False)
botonCNN = widgets.Button(description="Clasificar (CNN)", button_style="info",
                          icon="check", layout=widgets.Layout(width="220px", height="40px"))
salidaCNN = widgets.Output()

def procesarCNN(b):
    with salidaCNN:
        clear_output()
        if len(cargadorCNN.value) == 0:
            print("Primero sube una imagen, luego presiona Clasificar.")
            return
        try:
            valor = cargadorCNN.value
            contenido = (list(valor.values())[0]["content"] if isinstance(valor, dict)
                         else valor[0]["content"])
        except Exception as e:
            print(f"Error leyendo el archivo: {e}")
            return

        imgPil = Image.open(io.BytesIO(contenido)).convert("RGB")
        imgArray = np.array(imgPil)
        img = cv2.resize(imgArray, (224, 224)).astype("float32") / 255.0
        img = np.expand_dims(img, axis=0)

        probs = modeloCNN.predict(img, verbose=0)[0]
        idxClase = np.argmax(probs)
        claseIngles = clases[idxClase]
        claseEspanol = nombresEspanol[claseIngles]
        confianza = probs[idxClase] * 100
        info = infoLogistica[claseIngles]

        plt.figure(figsize=(5, 5))
        plt.imshow(imgPil); plt.axis("off")
        plt.title(f"Modelo CNN (MobileNetV2): {claseEspanol} ({confianza:.1f}%)", fontsize=13)
        plt.show()

        print("=" * 52)
        print(f"  MODELO CNN (MobileNetV2) -> {claseEspanol}  (confianza {confianza:.1f}%)")
        print("=" * 52)
        print(f"\n  Apto para consumo?  {info['apto']}")
        print(f"  Ventana de transporte: {info['ventana']}")
        print(f"  Mercado recomendado:   {info['mercado']}")
        print(f"  Observacion:           {info['obs']}")
        print(f"\n  Probabilidades por clase:")
        for i, c in enumerate(clases):
            barra = "#" * int(probs[i] * 25)
            print(f"    {nombresEspanol[c]:14s}: {probs[i]*100:5.1f}%  {barra}")

botonCNN.on_click(procesarCNN)
display(tituloCNN, cargadorCNN, botonCNN, salidaCNN)